# 1. Tải các thư viện cần thiết

In [1]:
!pip install underthesea openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 54.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.4 MB/s eta 0:00:00


# 2. Import thư viện

In [2]:
import pandas as pd
import numpy as np
import string
from underthesea import word_tokenize
from tqdm import tqdm
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
from gensim.models import KeyedVectors

print("[Thông báo] Đã import thư viện thành công.")

[Thông báo] Đã import thư viện thành công.


# 3. Trích xuất và Tiền xử lý dữ liệu

In [3]:
FILE_PATH = "/kaggle/input/datasets/qucvinhchu/fashionreviewslabeled/Balanced_Reviews.xlsx"
OUTPUT_EXCEL_PATH = "Tokenized_Reviews.xlsx"

# Bật thanh tiến trình cho pandas apply
tqdm.pandas(desc="Đang xử lý văn bản")

# 1. Định nghĩa các quy tắc làm sạch
# Xoá toàn bộ dấu câu nhưng giữ lại '%' (Ví dụ: 100% vải cotton)
punctuations = string.punctuation.replace('%', '').replace('_', '')

# Tạo danh sách các Stopword
# CHÚ Ý: Không chứa các từ phủ định (không, chưa, chẳng) hoặc từ chỉ mức độ (rất, quá, hơi)
custom_stopwords = set([
    "thì", "là", "mà", "của", "và", "với", "các", "những", 
    "cái", "chiếc", "này", "kia", "đó", "đây", "ấy", "sự", "việc", "như", "cho"
])

def preprocess_and_tokenize(text):
    if pd.isna(text): 
        return ""
    
    # Bước 1: Chuyển về chữ thường
    text = str(text).lower()
    
    # Bước 2: Tách từ tiếng Việt NGAY KHI CÒN DẤU CÂU
    # underthesea sẽ dùng dấu câu để nhận diện ranh giới ngắt từ chuẩn xác nhất
    tokenized_text = word_tokenize(text, format="text")
    
    # Bước 3: Loại bỏ dấu câu (thay bằng khoảng trắng)
    # Lưu ý: Các từ ghép tiếng Việt lúc này đã có dấu '_' (vd: giá_cả) nên không sợ bị đứt gãy
    clean_text = tokenized_text.translate(str.maketrans(punctuations, ' ' * len(punctuations)))
    
    # Bước 4: Lọc Stopword và các từ chỉ chứa số
    tokens = clean_text.split()
    clean_tokens = [
        word for word in tokens 
        if word not in custom_stopwords and not word.isnumeric()
    ]
    
    # Trả về chuỗi kết quả
    return " ".join(clean_tokens)

# 2. Đọc dữ liệu
print("[Thông báo] Đang thực hiện đọc dữ liệu Excel...")
df = pd.read_excel(FILE_PATH)
print("[Thông báo] Đã thực hiện xong việc đọc dữ liệu Excel.")

# 3. Áp dụng quy trình tiền xử lý lên cột "Nội dung review"
print("[Thông báo] Bắt đầu tiền xử lý và tách từ (quá trình này có thể mất vài phút)...")
df['Tokenized_Review'] = df['Nội dung review'].progress_apply(preprocess_and_tokenize)
print("[Thông báo] Đã thực hiện xong việc tiền xử lý và tách từ.")

# 4. Lưu kết quả ra file Excel (.xlsx)
# Lưu toàn bộ dataframe (bao gồm cả các cột nhãn ban đầu và cột Tokenized_Review mới)
print(f"[Thông báo] Đang lưu dữ liệu đã xử lý ra file: {OUTPUT_EXCEL_PATH}...")
df.to_excel(OUTPUT_EXCEL_PATH, index=False, engine='openpyxl')
print("[Thông báo] Đã thực hiện xong việc lưu dữ liệu.")

[Thông báo] Đang thực hiện đọc dữ liệu Excel...
[Thông báo] Đã thực hiện xong việc đọc dữ liệu Excel.
[Thông báo] Bắt đầu tiền xử lý và tách từ (quá trình này có thể mất vài phút)...


Đang xử lý văn bản: 100%|██████████| 95145/95145 [04:04<00:00, 388.80it/s]


[Thông báo] Đã thực hiện xong việc tiền xử lý và tách từ.
[Thông báo] Đang lưu dữ liệu đã xử lý ra file: Tokenized_Reviews.xlsx...
[Thông báo] Đã thực hiện xong việc lưu dữ liệu.


# 4. Xây dựng từ điển

In [24]:
FILE_PATH = "/kaggle/input/datasets/qucvinhchu/tokenized-reviewlabeled/Tokenized_Reviews.xlsx"

MIN_COUNT = 5

# 1. Đọc file dữ liệu đã được tiền xử lý
print(f"[Thông báo] Đang đọc dữ liệu từ {FILE_PATH}...")
df = pd.read_excel(FILE_PATH)
print("[Thông báo] Đã thực hiện xong việc đọc dữ liệu.")

# 2. Chuyển đổi cột chuỗi văn bản thành danh sách các token
# Ví dụ: "áo_đẹp giá_cả hợp_lý" --> ['áo_đẹp', 'giá_cả', 'hợp_lý']
print("[Thông báo] Đang chuyển đổi chuỗi thành danh sách các token...")
sentences = [str(text).split() for text in df['Tokenized_Review']]
print("[Thông báo] Đã thực hiện xong việc chuyển đổi chuỗi thành danh sách các token.")

# 3. Đếm tần suất xuất hiện của toàn bộ từ vựng
print("[Thông báo] Đang đếm tần suất từ vựng...")
word_counts = Counter(word for sentence in sentences for word in sentence)
total_unique_words_before = len(word_counts)
print("[Thông báo] Đã thực hiện xong việc đếm tần từ vựng.")
print(f"[Kết quả] Tổng số từ vựng duy nhất (trước khi lọc): {total_unique_words_before}")

# 4. Lọc các từ hiếm (chỉ giữ lại các từ xuất hiện >= MIN_COUNT)
print("[Thông báo] Đang thực hiện lọc các từ hiếm...")
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]
print("[Thông báo] Đã thực hiện xong việc lọc các từ hiếm.")

# 5. Thêm 2 token đặc biệt vào đầu từ điển
vocab.insert(0, '<PAD>') # Luôn nằm ở index 0
vocab.insert(1, '<UNK>') # Luôn nằm ở index 1

vocab_size = len(vocab)
print(f"Tổng số từ vựng (sau khi lọc MIN_COUNT={MIN_COUNT}): {vocab_size}")

# 6. Tạo 2 từ điển ánh xạ (Mappings)
# word2idx: Truyền vào 1 từ --> Trả về ID số nguyên (dùng lúc đưa vào model)
word2idx = {word: idx for idx, word in enumerate(vocab)}

# idx2word: Truyền vào 1 ID --> Trả về từ gốc (dùng lúc lưu model hoặc test kết quả)
idx2word = {idx: word for word, idx in word2idx.items()}

[Thông báo] Đang đọc dữ liệu từ /kaggle/input/datasets/qucvinhchu/tokenized-reviewlabeled/Tokenized_Reviews.xlsx...
[Thông báo] Đã thực hiện xong việc đọc dữ liệu.
[Thông báo] Đang chuyển đổi chuỗi thành danh sách các token...
[Thông báo] Đã thực hiện xong việc chuyển đổi chuỗi thành danh sách các token.
[Thông báo] Đang đếm tần suất từ vựng...
[Thông báo] Đã thực hiện xong việc đếm tần từ vựng.
[Kết quả] Tổng số từ vựng duy nhất (trước khi lọc): 16733
[Thông báo] Đang thực hiện lọc các từ hiếm...
[Thông báo] Đã thực hiện xong việc lọc các từ hiếm.
Tổng số từ vựng (sau khi lọc MIN_COUNT=5): 5362


In [26]:
# In thử một vài từ trong từ điển để kiểm tra
print("\n--- Kiểm tra Từ điển ---")
sample_words = ['<PAD>', '<UNK>', 'chất_liệu', 'form', 'rộng', 'áo', "tốt"]
for w in sample_words:
    if w in word2idx:
        print(f"Từ '{w}' có ID là: {word2idx[w]}")
    else:
        print(f"Từ '{w}' không có trong từ điển (sẽ được gán thành <UNK>).")


--- Kiểm tra Từ điển ---
Từ '<PAD>' có ID là: 0
Từ '<UNK>' có ID là: 1
Từ 'chất_liệu' có ID là: 108
Từ 'form' có ID là: 100
Từ 'rộng' có ID là: 32
Từ 'áo' có ID là: 2
Từ 'tốt' có ID là: 112


# 5. Tạo các cặp dữ liệu huấn luyện và bộ nạp lô dữ liệu cho CBOW

In [27]:
# 1. Định nghĩa kích thước cửa sổ ngữ cảnh
# Trong đánh giá thời trang, các tính từ mô tả (vd: rộng, mát) thường cách danh từ (vd: áo, vải) khoảng 2-5 từ.
WINDOW_SIZE = 5

class FashionCBOWDataset(Dataset):
    def __init__(self, sentences, word2idx, window_size):
        self.data = []
        print("[Thông báo] Đang trượt cửa sổ tạo các cặp (Ngữ cảnh, Mục tiêu)...")

        for sentence in sentences:
            # Chuyển đổi toàn bộ từ trong câu thành ID. Nếu từ không có trong từ điển, gán thành <UNK>
            indices = [word2idx.get(word, word2idx['<UNK>']) for word in sentence]

            for i, target in enumerate(indices):
                # Cắt lấy từ bên trái và bên phải từ mục tiêu
                context = (
                    indices[max(0, i - window_size):i] +
                    indices[i + 1:min(len(indices), i + window_size + 1)]
                )

                # Nếu từ nằm ở đầu hoặc cuối câu, ngữ cảnh sẽ bị ngắn. Cần chèn thêm <PAD> (ID=0)
                while len(context) < 2 * window_size:
                    context.append(word2idx['<PAD>'])

                # Lưu lại cặp (context, target)
                self.data.append((context, target))
        print("[Thông báo] Đã thực hiện xong việc trượt cửa sổ tạo các cặp (Ngữ cảnh, Mục tiêu).")
        print(f"[Kết quả] Đã tạo ra {len(self.data)} mẫu dữ liệu huấn luyện.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        # Chuyển đổi thành Pytorch Tensor (Kiểu số nguyên long)
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

# 2. Khởi tạo Dataset
print("[Thông báo] Đang khởi tạo dataset cho CBOW...")
dataset = FashionCBOWDataset(sentences, word2idx, WINDOW_SIZE)
print("[Thông báo] Đã thực hiện xong việc khởi tạo dataset cho CBOW.")

# 3. Khởi tạo Dataloader
# Để tận dụng tối đa sức mạnh của 2 GPU T4, việc chọn BATCH_SIZE lớn là rất quan trọng.
BATCH_SIZE = 2048
# num_workers giúp nạp dữ liệu song song từ CPU đẩy lên GPU nhanh hơn
num_workers = os.cpu_count() if os.cpu_count() else 2

print(f"[Thông báo] Đang thiết lập DataLoader với Batch Size = {BATCH_SIZE} và {num_workers} workers...")
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, # Trộn đều dữ liệu để mô hình học tốt hơn
    num_workers=num_workers,
    pin_memory=True # Giúp truyền dữ liệu từ RAM lên VRAM của GPU nhanh hơn
)
print("[Thông báo] Đã thực hiện xong việc thiết lập DataLoader.")

[Thông báo] Đang khởi tạo dataset cho CBOW...
[Thông báo] Đang trượt cửa sổ tạo các cặp (Ngữ cảnh, Mục tiêu)...
[Thông báo] Đã thực hiện xong việc trượt cửa sổ tạo các cặp (Ngữ cảnh, Mục tiêu).
[Kết quả] Đã tạo ra 2856696 mẫu dữ liệu huấn luyện.
[Thông báo] Đã thực hiện xong việc khởi tạo dataset cho CBOW.
[Thông báo] Đang thiết lập DataLoader với Batch Size = 2048 và 4 workers...
[Thông báo] Đã thực hiện xong việc thiết lập DataLoader.


# 6. Định nghĩa Kiến trúc Mạng Neural CBOW và kích hoạt huấn luyện mô hình.

In [28]:
# 1. Định nghĩa kiến trúc mô hình CBOW
print("[Thông báo] Đang định nghĩa kiến trúc Mạng Neural CBOW...")
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        # Lớp Embedding: Biến ID thành vector. padding_idx=0 để hệ thống bỏ qua token <PAD>
        self.embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Lớp Linear: Lấy vector tổng hợp để dự đoán xem từ mục tiêu (Target) là từ nào trong từ điển
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_words):
        # 1. Tra bảng để lấy vector của các từ ngữ cảnh xung quanh
        embeds = self.embeddings(context_words)

        # 2. Thuật toán CBOW: Lấy trung bình cộng (mean) của các vector ngữ cảnh này
        mean_embeds = torch.mean(embeds, dim=1)

        # 3. Đưa qua lớp tuyến tính để tính xác suất (logits) cho từng từ trong từ điển
        out = self.linear(mean_embeds)
        return out
print("[Thông báo] Đã thực hiện xong việc định nghĩa kiến trúc Mạng Neural CBOW.")

[Thông báo] Đang định nghĩa kiến trúc Mạng Neural CBOW...
[Thông báo] Đã thực hiện xong việc định nghĩa kiến trúc Mạng Neural CBOW.


In [31]:
# 2. Cấu hình tham số mô hình và đa GPU
print("[Thông báo] Đang cấu hình tham số mô hình và đa GPU...")
# Cấu hình hyperparameters
EMBEDDING_DIM = 100 # Số chiều của vector (150 là mức rất tốt để biểu diễn các khía cạnh thời trang)
EPOCHS = 50 # Số lần duyệt qua toàn bộ bộ dữ liệu
LEARNING_RATE = 0.005 # Tốc độ học (để hơi cao một chút vì ta đang dùng Batch Size lớn)

# Khởi tạo mô hình
model = CBOWModel(vocab_size, EMBEDDING_DIM)

# Kiểm tra thiết bị (Device)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# BẬT CHẾ ĐỘ MULTI-GPU TRÊN KAGGLE
if torch.cuda.device_count() > 1:
    print(f"Tuyệt vời! Đang kích hoạt nn.DataParallel để phân tải lên {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

# Đẩy mô hình lên GPU
model.to(device)

# Hàm tính lỗi (Loss Function): Bỏ qua lỗi của token <PAD> (ID=0)
criterion = nn.CrossEntropyLoss(ignore_index=0)
# Bộ tối ưu hóa (Optimizer): AdamW hoạt động cực kỳ hiệu quả với bài toán NLP
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print("[Thông báo] Đã thực hiện xong việc cấu hình tham số mô hình và đa GPU.")

[Thông báo] Đang cấu hình tham số mô hình và đa GPU...
Tuyệt vời! Đang kích hoạt nn.DataParallel để phân tải lên 2 GPUs!
[Thông báo] Đã thực hiện xong việc cấu hình tham số mô hình và đa GPU.


In [32]:
# 3. Thực hiện huấn luyện mô hình
print("[Thông báo] Bắt đầu quá trình huấn luyện song song...")
for epoch in range(EPOCHS):
    total_loss = 0
    model.train() # Đặt mô hình ở chế độ huấn luyện
    
    # Thanh tiến trình trực quan
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    
    for context, target in progress_bar:
        # Đẩy mẻ dữ liệu từ RAM lên bộ nhớ của 2 GPU
        context, target = context.to(device), target.to(device)
        
        # Xóa các tính toán gradient cũ
        optimizer.zero_grad()
        
        # Chạy tiến (Forward): Dự đoán từ
        output = model(context)
        
        # Tính sai số (Loss)
        loss = criterion(output, target)
        
        # Chạy lùi (Backward): Tính toán đạo hàm
        loss.backward()
        
        # Cập nhật trọng số của ma trận Embedding
        optimizer.step()
        
        total_loss += loss.item()
        
        # Cập nhật thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} hoàn tất | Average Loss: {avg_loss:.4f}")
print("[Thông báo] Hoàn tất quá trình huấn luyện song song.")

[Thông báo] Bắt đầu quá trình huấn luyện song song...


Epoch 1/50: 100%|██████████| 1395/1395 [00:29<00:00, 46.66it/s, loss=3.9172]


Epoch 1 hoàn tất | Average Loss: 4.4491


Epoch 2/50: 100%|██████████| 1395/1395 [00:27<00:00, 50.85it/s, loss=3.6051]


Epoch 2 hoàn tất | Average Loss: 3.7946


...

Epoch 48 hoàn tất | Average Loss: 3.3671


Epoch 49/50: 100%|██████████| 1395/1395 [00:27<00:00, 51.44it/s, loss=3.3488]


Epoch 49 hoàn tất | Average Loss: 3.3667


Epoch 50/50: 100%|██████████| 1395/1395 [00:27<00:00, 50.87it/s, loss=3.4430]

Epoch 50 hoàn tất | Average Loss: 3.3666
[Thông báo] Hoàn tất quá trình huấn luyện song song.


In [33]:
# 4. Lưu lại trọng số của mô hình
print("[Thông báo] Đang trích xuất ma trận Embedding...")
# Lấy ma trận trọng số
if isinstance(model, nn.DataParallel):
    embedding_matrix = model.module.embeddings.weight.data.cpu().numpy()
else:
    embedding_matrix = model.embeddings.weight.data.cpu().numpy()

output_file = "/kaggle/working/fashion_word2vec.txt"

# BƯỚC SỬA LỖI: Tính toán số lượng từ thực tế sẽ được lưu vào file
words_to_skip = ['<PAD>', '<UNK>']
actual_vocab_size = vocab_size - len(words_to_skip)

print(f"Đang lưu file tại: {output_file}...")
with open(output_file, "w", encoding="utf-8") as f:
    # Ghi số lượng từ THỰC TẾ và số chiều vector
    f.write(f"{actual_vocab_size} {EMBEDDING_DIM}\n")
    
    for idx in range(vocab_size):
        word = idx2word[idx]
        if word in words_to_skip: 
            continue
            
        vec_str = " ".join(map(str, embedding_matrix[idx]))
        f.write(f"{word} {vec_str}\n")
print("[Thông báo] Đã thực hiện xong việc trích xuất ma trận Embedding.")

[Thông báo] Đang trích xuất ma trận Embedding...
Đang lưu file tại: /kaggle/working/fashion_word2vec.txt...
[Thông báo] Đã thực hiện xong việc trích xuất ma trận Embedding.


# 7. Đánh giá chất lượng của mô hình

In [37]:
print("[Thông báo] Đang nạp mô hình vào Gensim để kiểm thử...")
# Load file text vừa tạo
word_vectors = KeyedVectors.load_word2vec_format('/kaggle/input/datasets/qucvinhchu/cbow-word2vec-100/fashion_word2vec.txt', binary=False)
print("[Thông báo] Đã thực hiện xong việc nạp mô hình vào Gensim để kiểm thử...")

# Viết một hàm nhỏ để test
def test_similarity(word):
    print(f"\n--- Các từ liên quan nhất đến '{word}' ---")
    try:
        # Tìm top 5 từ có vector gần nhất (Cosine Similarity)
        similar_words = word_vectors.most_similar(word, topn=5)
        for w, score in similar_words:
            print(f" + {w}: {score:.4f}")
    except KeyError:
        print(f"Từ '{word}' không có trong từ điển.")

# BẮT ĐẦU TEST (Bạn có thể thay đổi các từ này theo ý muốn)
test_similarity('rộng')        
test_similarity('chất_liệu')   
test_similarity('giao')   
test_similarity('giá_cả')
test_similarity('áo')
test_similarity('quần_ngắn')
test_similarity('dịch_vụ')
test_similarity('tốt')
test_similarity('tệ')
test_similarity('quần_jeans')
test_similarity('size')

[Thông báo] Đang nạp mô hình vào Gensim để kiểm thử...

--- Các từ liên quan nhất đến 'rộng' ---
 + ngắn: 0.5813
 + lỏng: 0.5550
 + to: 0.5321
 + nhỏ: 0.5256
 + hẹp: 0.5033

--- Các từ liên quan nhất đến 'chất_liệu' ---
 + chất: 0.8224
 + _chất: 0.4027
 + chất_cotton: 0.3915
 + lọ_chất: 0.3902
 + chất_liệu_dạ: 0.3891

--- Các từ liên quan nhất đến 'giao' ---
 + ship: 0.5789
 + vận_chuyển_giao: 0.5561
 + chuẩn_bị: 0.5507
 + gửi: 0.5225
 + chuẩn_bị_đơn: 0.4864

--- Các từ liên quan nhất đến 'giá_cả' ---
 + giá_thành: 0.9137
 + giá: 0.7783
 + được_giá: 0.5013
 + có_giá: 0.4258
 + m2: 0.3641

--- Các từ liên quan nhất đến 'áo' ---
 + váy: 0.5953
 + quần_áo: 0.5510
 + khăn: 0.5262
 + áo_khoác: 0.5165
 + áo_phông: 0.5156

--- Các từ liên quan nhất đến 'quần_ngắn' ---
 + quần_lửng: 0.3859
 + quần: 0.3840
 + đứng: 0.3764
 + tự_tin: 0.3741
 + loại: 0.3549

--- Các từ liên quan nhất đến 'dịch_vụ' ---
 + quy_trình: 0.4872
 + khâu: 0.4544
 + đội_ngũ: 0.4330
 + tốc_độ: 0.4229
 + quá_trình: 0.4175



# 8. Huấn luyện mô hình FastText sử dụng kiến trúc CBOW

In [1]:
from gensim.models import KeyedVectors
from gensim.models import FastText
from gensim.models.callbacks import CallbackAny2Vec
from tqdm import tqdm
import pandas as pd

In [14]:
# 1. Định nghĩa Callback cho thanh tiến trình
class EpochLogger(CallbackAny2Vec):
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
        self.epoch = 0
        # Khởi tạo thanh tiến trình tqdm
        self.pbar = tqdm(total=self.total_epochs, desc="Đang huấn luyện FastText")

    def on_epoch_end(self, model):
        self.epoch += 1
        self.pbar.update(1) # Tiến lên 1 bước sau mỗi epoch
        
        # Đóng thanh tiến trình khi hoàn tất
        if self.epoch == self.total_epochs:
            self.pbar.close()

# 2. Chuẩn bị dữ liệu
FILE_PATH = "/kaggle/input/datasets/qucvinhchu/tokenized-reviewlabeled/Tokenized_Reviews.xlsx"
df = pd.read_excel(FILE_PATH)

print("[Thông báo] Đang chuẩn bị dữ liệu...")
sentences = [str(text).split() for text in df['Tokenized_Review'].tolist()]
print("[Thông báo] Đã thực hiện xong việc chuẩn bị dữ liệu.")

# Huấn luyện mô hình
print("[Thông báo] Bắt đầu huấn luyện FastText từ đầu (From scratch)...")

EPOCHS = 50
# Khởi tạo đối tượng theo dõi tiến trình
progress_callback = EpochLogger(total_epochs=EPOCHS)

# Khởi tạo và huấn luyện mô hình FastText
ft_model = FastText(
    sentences=sentences, 
    vector_size=100,      
    window=5,             
    min_count=5,          
    sg=0,                 
    min_n=3, 
    max_n=6,
    bucket=200000,
    workers=4,            
    epochs=EPOCHS,             
    callbacks=[progress_callback]
)
ft_model.callbacks = ()
print("[Thông báo] Đã thực hiện xong việc huấn luyện FastText từ đầu (From scratch).")

# Lưu mô hình
print("[Thông báo] Đang tách riêng Lõi Vector (KeyedVectors)...")
# Lệnh này lưu phần lõi có khả năng tính toán sub-words
ft_model.wv.save("/kaggle/working/fashion_fasttext_core.kv")
print("[Thông báo] Đã thực hiện xong việc tách riêng Lõi Vector (KeyedVectors).")
print("\n✅ Huấn luyện thành công! Mô hình đã xử lý được lỗi chính tả/từ lóng.")

[Thông báo] Đang chuẩn bị dữ liệu...
[Thông báo] Đã thực hiện xong việc chuẩn bị dữ liệu.
[Thông báo] Bắt đầu huấn luyện FastText từ đầu (From scratch)...


Đang huấn luyện FastText: 100%|██████████| 50/50 [07:34<00:00,  9.10s/it]


[Thông báo] Đã thực hiện xong việc huấn luyện FastText từ đầu (From scratch).
[Thông báo] Đang tách riêng Lõi Vector (KeyedVectors)...
[Thông báo] Đã thực hiện xong việc tách riêng Lõi Vector (KeyedVectors).

✅ Huấn luyện thành công! Mô hình đã xử lý được lỗi chính tả/từ lóng.


In [16]:
# Đường dẫn tới file .kv của bạn (thay đổi nếu bạn lưu ở nơi khác)
KV_FILE_PATH = "/kaggle/input/datasets/qucvinhchu/cbowfasttext-word2vec-100/fashion_fasttext_core.kv"

print("[Thông báo] Đang nạp lõi vector (KeyedVectors)...")
# Dùng hàm load() cơ bản cho định dạng .kv
word_vectors = KeyedVectors.load(KV_FILE_PATH)

# Viết một hàm nhỏ để test
def test_similarity(word):
    print(f"\n--- Các từ liên quan nhất đến '{word}' ---")
    try:
        # Tìm top 5 từ có vector gần nhất (Cosine Similarity)
        similar_words = word_vectors.most_similar(word, topn=5)
        for w, score in similar_words:
            print(f" + {w}: {score:.4f}")
    except KeyError:
        print(f"Từ '{word}' không có trong từ điển.")

# BẮT ĐẦU TEST (Bạn có thể thay đổi các từ này theo ý muốn)
test_similarity('rộnggggn')        
test_similarity('chất')   
test_similarity('giao_hànggg')   
test_similarity('giá_cả')
test_similarity('áo')
test_similarity('quần_ngắn')
test_similarity('dịch_vụ')
test_similarity('tốt')
test_similarity('tệ')
test_similarity('quần_jeans')
test_similarity('size')

[Thông báo] Đang nạp lõi vector (KeyedVectors)...

--- Các từ liên quan nhất đến 'rộnggggn' ---
 + rộng: 0.9427
 + rộng_rãi: 0.7407
 + rộng_xíu: 0.7376
 + mở_rộng: 0.6676
 + ôm_sát: 0.5887

--- Các từ liên quan nhất đến 'chất' ---
 + chất_liệu: 0.8761
 + chất_liệu_dạ: 0.8378
 + chất_liệu_su: 0.8283
 + _chất: 0.8163
 + chất_vải: 0.8107

--- Các từ liên quan nhất đến 'giao_hànggg' ---
 + giao_đồ: 0.8284
 + giao_lẹ: 0.8257
 + giao_lộn: 0.8182
 + giao_nhận: 0.8017
 + giao_muộn: 0.7996

--- Các từ liên quan nhất đến 'giá_cả' ---
 + giá_thành: 0.9472
 + giá_mà: 0.9019
 + giáo: 0.8142
 + giá_rét: 0.7962
 + giá: 0.7134

--- Các từ liên quan nhất đến 'áo' ---
 + áo_ấm: 0.7326
 + áo_sơ: 0.7307
 + áo_ok: 0.7291
 + áo_yếm: 0.6989
 + áo_quần: 0.6656

--- Các từ liên quan nhất đến 'quần_ngắn' ---
 + quần_lửng: 0.7982
 + quần_ôm: 0.7913
 + quần_co: 0.7878
 + quần_ag: 0.7852
 + quần_boyfriend: 0.7813

--- Các từ liên quan nhất đến 'dịch_vụ' ---
 + dịch_bệnh: 0.8123
 + vụ: 0.7744
 + nhân_viên: 0.6493
 